In [10]:
%pip install docling
%pip install boto3


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
    WordFormatOption,
)
from docling.pipeline.simple_pipeline import SimplePipeline
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline

import logging
import os
import json
from dotenv import load_dotenv
import boto3
from botocore.client import Config
from boto3 import session
import tempfile
from pathlib import Path
import glob


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

load_dotenv()
file_type=os.getenv("FILE_TYPE",".TXT")
file_source=os.getenv("FILE_SOURCE_LOCATION","/home/noelo/")
md_destination=os.getenv("MARKDOWN_LOCATION","file:///tmp")

if file_type.lower() not in ".pdf .docx .odf .txt":
    raise Exception("Invalid or empty file type. Only PDF, DOCX or ODF files supported. Set in FILE_TYPE envar")

if not file_source:
    raise Exception("Invalid or empty file source location. Set in FILE_SOURCE_LOCATION")

if not md_destination:
    raise Exception("Invalid or empty file source location. Set in MARKDOWN_LOCATION")

tmpdir = None
file_list=[]

.TXT


Figure out what source of files we're dealing with and then list and filter them. Returning a list of files that we need to process.

In [56]:
def filter_file_ext(filename) -> bool:
    file_name, file_extension = os.path.splitext(filename)

    if not file_extension:
        return False
    
    if file_extension.lower().strip() in file_type.lower():
        print(f"{file_name}:[{file_extension}]")
        return True
    else:
        return False

In [57]:
if "s3://" in file_source:
    # Get file list from s3
    key_id = os.environ.get('AWS_ACCESS_KEY_ID')
    secret_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
    endpoint = os.environ.get('AWS_S3_ENDPOINT')
    region = os.environ.get('AWS_DEFAULT_REGION')
    session = boto3.session.Session(aws_access_key_id=key_id,
    aws_secret_access_key=secret_key)

    #Define client connection
    s3_client = boto3.client('s3', aws_access_key_id=key_id,
    aws_secret_access_key=secret_key,aws_session_token=None,
        config=boto3.session.Config(signature_version='s3v4'),
                            endpoint_url=endpoint,
                            region_name=region)
    bucket_name = file_source.split("s3://")[1]
    print(bucket_name)
    
    default_kwargs = {
        "Bucket": bucket_name,
    }
    
    response = s3_client.list_objects_v2(**default_kwargs)
    contents = response.get("Contents")
    
    tempfile.TemporaryDirectory()
    print(f'Temporary directory created at: {tmpdir.name}')
    
    for result in contents:
        file_name = result.get("Key")
        file_path = Path.joinpath(tmpdir.name, file_name)
        
        s3_client.download_file(
            bucket_name,
            file_name,
            str(file_path)
        )
        
        file_list.append(str(file_path))
        
    file_source = tmpdir.name
else:
    for file in glob.iglob(file_source+"/*", recursive=False):
        file_path = Path.joinpath(Path(file_source), file)
        file_list.append(str(file_path))
  
filtered_files = filter(filter_file_ext,file_list)
print(list(filtered_files))
     

/home/noelo/all-off:[.txt]
/home/noelo/all-day:[.txt]
/home/noelo/t1:[.txt]
/home/noelo/t2:[.txt]
/home/noelo/all-auto:[.txt]
/home/noelo/t3:[.txt]
['/home/noelo/all-off.txt', '/home/noelo/all-day.txt', '/home/noelo/t1.txt', '/home/noelo/t2.txt', '/home/noelo/all-auto.txt', '/home/noelo/t3.txt']


In [ ]:
doc_converter = (
    DocumentConverter(  # all of the below is optional, has internal defaults.
        allowed_formats=[
            InputFormat.PDF,
            InputFormat.DOCX,
        ],  # whitelist formats, non-matching files are ignored.
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_cls=StandardPdfPipeline, backend=PyPdfiumDocumentBackend
            ),
            InputFormat.DOCX: WordFormatOption(
                pipeline_cls=SimplePipeline  # , backend=MsWordDocumentBackend
            ),
        },
    )
)

In [ ]:
from quackling.llama_index.node_parsers import HierarchicalJSONNodeParser
from quackling.llama_index.readers import DoclingPDFReader

reader = DoclingPDFReader(parse_type=DoclingPDFReader.ParseType.JSON)
node_parser = HierarchicalJSONNodeParser()

In [ ]:
docs = reader.load_data(file_path=source)

In [ ]:
from rich.pretty import pprint
pprint(docs, max_length=2, max_string=250, max_depth=4)